# Lab 12: Generative AI — Fundamentals & Prompt Engineering
## Objective
Understand the fundamentals of Generative AI and experiment with prompt-based tools. Explore key concepts such as language model architecture, prompt engineering techniques, temperature and sampling parameters, and practical applications.

**Concepts Covered:**
- What is Generative AI?
- Types of Generative Models (GANs, VAEs, Transformers)
- Transformer Architecture & Attention Mechanism
- Prompt Engineering Techniques
- Experimenting with Text Generation
- Temperature & Sampling Parameters
- Applications of Generative AI

## 1. Import Libraries

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

np.random.seed(42)

print("All libraries loaded successfully.")

All libraries loaded successfully.


## 2. What is Generative AI?

Generative AI refers to artificial intelligence systems that can **generate new content** — such as text, images, audio, code, and video — based on patterns learned from training data.

### Key Characteristics:
- **Creates new content** rather than just classifying or predicting
- **Learns patterns** from large datasets during training
- **Produces diverse outputs** that resemble the training data distribution

### Common Types of Generative Models:

| Model Type | Description | Examples |
|:---|:---|:---|
| **GANs** (Generative Adversarial Networks) | Two networks (generator & discriminator) compete against each other | StyleGAN, DCGAN |
| **VAEs** (Variational Autoencoders) | Encoder-decoder architecture with a latent space | β-VAE |
| **Transformers** | Self-attention based architecture for sequential data | GPT, BERT, T5 |
| **Diffusion Models** | Iteratively denoise random noise into structured data | Stable Diffusion, DALL-E 2 |

In [2]:
# Visualization: Timeline of Generative AI Milestones
milestones = {
    '2014': 'GANs\n(Goodfellow)',
    '2017': 'Transformer\n(Vaswani et al.)',
    '2018': 'GPT-1\n(OpenAI)',
    '2019': 'GPT-2\n(OpenAI)',
    '2020': 'GPT-3\n(175B params)',
    '2022': 'ChatGPT &\nStable Diffusion',
    '2023': 'GPT-4 &\nGemini',
    '2024': 'Multimodal\nAI Era'
}

fig, ax = plt.subplots(figsize=(14, 6))
years = list(milestones.keys())
labels = list(milestones.values())
x_pos = np.arange(len(years))

colors = plt.cm.plasma(np.linspace(0.15, 0.85, len(years)))

for i, (x, year, label) in enumerate(zip(x_pos, years, labels)):
    ax.scatter(x, 0, s=200, color=colors[i], zorder=5, edgecolors='white', linewidth=2)
    offset = 0.4 if i % 2 == 0 else -0.5
    ax.annotate(f'{year}\n{label}', xy=(x, 0), xytext=(x, offset),
                fontsize=10, ha='center', va='center',
                bbox=dict(boxstyle='round,pad=0.4', facecolor=colors[i], alpha=0.3),
                arrowprops=dict(arrowstyle='->', color=colors[i], lw=1.5))

ax.plot(x_pos, [0]*len(x_pos), '--', color='gray', alpha=0.5, linewidth=2)
ax.set_ylim(-1.0, 1.0)
ax.set_xlim(-0.5, len(years) - 0.5)
ax.set_title('Timeline of Major Generative AI Milestones', fontsize=16, fontweight='bold', pad=20)
ax.axis('off')
plt.tight_layout()
plt.show()

## 3. Transformer Architecture Overview

The **Transformer** architecture (introduced in "Attention Is All You Need", Vaswani et al., 2017) is the foundation of modern Generative AI.

### Key Components:
1. **Self-Attention Mechanism** — Allows the model to weigh the importance of different tokens in a sequence
2. **Multi-Head Attention** — Multiple attention heads capture different relationships
3. **Positional Encoding** — Injects sequence order information
4. **Feed-Forward Networks** — Per-position fully connected layers
5. **Layer Normalization** — Stabilizes training

In [3]:
print("=== SELF-ATTENTION MECHANISM (Simplified) ===")

# Simulate self-attention with random embeddings
tokens = ['The', 'wheat', 'seed', 'grows', 'well']
d_model = 8  # Small embedding dimension for demo

# Random embeddings for each token
embeddings = np.random.randn(len(tokens), d_model)

# Compute Q, K, V (simplified: using the embeddings directly)
Q = embeddings  # Query
K = embeddings  # Key
V = embeddings  # Value

# Attention scores: softmax(Q @ K^T / sqrt(d_k))
d_k = d_model
scores = Q @ K.T / np.sqrt(d_k)

# Softmax
def softmax(x, axis=-1):
    e_x = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return e_x / e_x.sum(axis=axis, keepdims=True)

attention_weights = softmax(scores)

print(f"\nInput tokens: {tokens}")
print(f"\nAttention Scores (how much each token attends to others):")
print(f"\n{'':>8}", end='')
for t in tokens:
    print(f"{t:>8}", end='')
print()
for i, t in enumerate(tokens):
    print(f"{t:>8}", end='')
    for j in range(len(tokens)):
        print(f"{attention_weights[i][j]:>8.3f}", end='')
    print()

=== SELF-ATTENTION MECHANISM (Simplified) ===

Input tokens: ['The', 'wheat', 'seed', 'grows', 'well']

Attention Scores (how much each token attends to others):

            The   wheat    seed   grows    well
  The     0.183   0.197   0.215   0.203   0.202
  wheat   0.203   0.173   0.198   0.206   0.220
  seed    0.207   0.200   0.197   0.208   0.188
  grows   0.142   0.209   0.226   0.210   0.213
  well    0.222   0.204   0.189   0.177   0.209


In [4]:
# Visualize Attention Weights
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(attention_weights, cmap='YlOrRd', aspect='auto')

ax.set_xticks(range(len(tokens)))
ax.set_yticks(range(len(tokens)))
ax.set_xticklabels(tokens, fontsize=12)
ax.set_yticklabels(tokens, fontsize=12)
ax.set_xlabel('Key Tokens', fontsize=13)
ax.set_ylabel('Query Tokens', fontsize=13)
ax.set_title('Self-Attention Weights Heatmap', fontsize=15, fontweight='bold')

# Add text annotations
for i in range(len(tokens)):
    for j in range(len(tokens)):
        ax.text(j, i, f'{attention_weights[i][j]:.3f}',
                ha='center', va='center', fontsize=10,
                color='white' if attention_weights[i][j] > 0.25 else 'black')

plt.colorbar(im, ax=ax, label='Attention Weight')
plt.tight_layout()
plt.show()

## 4. Prompt Engineering Techniques

Prompt engineering is the art and science of designing inputs (prompts) to guide AI models toward desired outputs.

### Key Techniques:

| Technique | Description | Example |
|:---|:---|:---|
| **Zero-shot** | No examples provided | "Classify this text as positive or negative" |
| **Few-shot** | A few examples included in the prompt | "Positive: Great movie! Negative: Terrible film. Classify: ..." |
| **Chain-of-Thought** | Step-by-step reasoning | "Let's think step by step..." |
| **Role-based** | Assign a persona | "You are an expert data scientist..." |
| **Instruction-based** | Clear, specific instructions | "Summarize in 3 bullet points..." |

In [5]:
print("=== PROMPT ENGINEERING EXAMPLES ===")

prompts = {
    "Zero-Shot Prompting": {
        "prompt": '''Classify the following text as 'Positive', 'Negative', or 'Neutral'.

Text: "The wheat seeds from this batch show excellent germination rates."

Classification:''',
        "description": "No examples given. The model relies entirely on its\n    pre-trained knowledge to understand and execute the task."
    },
    "Few-Shot Prompting": {
        "prompt": '''Classify the following texts:

Text: "The crop yield was outstanding this season." → Positive
Text: "The seeds failed to germinate properly." → Negative
Text: "The temperature was 25 degrees today." → Neutral

Text: "These wheat kernels have superior quality measurements." →''',
        "description": "A few labeled examples are provided to guide the\n    model's understanding of the task format and expected output."
    },
    "Chain-of-Thought Prompting": {
        "prompt": '''A farmer has 3 fields. Field A produces 150 kg of wheat, Field B produces
twice as much as Field A, and Field C produces 50 kg less than Field B.
What is the total wheat production?

Let's solve this step by step:
1. Field A produces 150 kg
2. Field B produces 2 × 150 = 300 kg
3. Field C produces 300 - 50 = 250 kg
4. Total = 150 + 300 + 250 = 700 kg''',
        "description": "The prompt encourages step-by-step reasoning,\n    which helps the model arrive at more accurate conclusions."
    },
    "Role-Based Prompting": {
        "prompt": '''You are an expert agricultural data scientist with 20 years of experience.
Analyze the following wheat kernel measurements and provide insights:

- Area: 15.26, Perimeter: 14.84, Compactness: 0.871
- Kernel Length: 5.763, Kernel Width: 3.312

What variety of wheat does this likely belong to?''',
        "description": "By assigning a specific role/persona, the model\n    generates more specialized and expert-level responses."
    },
    "Instruction-Based Prompting": {
        "prompt": '''Instructions:
1. Read the following paragraph about wheat classification.
2. Extract the key features mentioned.
3. List them as bullet points.
4. Keep each bullet point under 10 words.

Paragraph: "Wheat kernels can be classified by measuring their area,
perimeter, compactness, kernel length, kernel width, asymmetry
coefficient, and groove length."''',
        "description": "Clear, structured instructions help the model\n    produce well-formatted and precise outputs."
    }
}

for name, data in prompts.items():
    print(f"\n\n{'═' * 62}")
    print(f"  TECHNIQUE: {name}")
    print(f"{'═' * 62}")
    print(f"  {data['prompt']}")
    print(f"\n  → Description: {data['description']}")

=== PROMPT ENGINEERING EXAMPLES ===


══════════════════════════════════════════════════════════════
  TECHNIQUE: Zero-Shot Prompting
══════════════════════════════════════════════════════════════
  Classify the following text as 'Positive', 'Negative', or 'Neutral'.

Text: "The wheat seeds from this batch show excellent germination rates."

Classification:

  → Description: No examples given. The model relies entirely on its
    pre-trained knowledge to understand and execute the task.


══════════════════════════════════════════════════════════════
  TECHNIQUE: Few-Shot Prompting
══════════════════════════════════════════════════════════════
  Classify the following texts:

Text: "The crop yield was outstanding this season." → Positive
Text: "The seeds failed to germinate properly." → Negative
Text: "The temperature was 25 degrees today." → Neutral

Text: "These wheat kernels have superior quality measurements." →

  → Description: A few labeled examples are provided to guide the
   

## 5. Temperature & Sampling Parameters

In [6]:
print("=== EFFECT OF TEMPERATURE ON SAMPLING ===")

def softmax_with_temperature(logits, temperature=1.0):
    """Apply temperature scaling to logits and compute softmax."""
    scaled = np.array(logits) / temperature
    exp_scaled = np.exp(scaled - np.max(scaled))
    return exp_scaled / exp_scaled.sum()

# Simulated logits for next token prediction
logits = [2.0, 1.5, 0.5, 0.3, 0.1]
vocab = ['wheat', 'seed', 'crop', 'field', 'soil']
temperatures = [0.1, 0.5, 1.0, 1.5, 2.0]

print(f"\nRaw logits: {logits}")
print(f"Vocabulary: {vocab}")

for temp in temperatures:
    probs = softmax_with_temperature(logits, temp)
    behavior = 'Very deterministic' if temp <= 0.1 else ('Very random' if temp >= 2.0 else f'→ {"Focused" if temp < 1 else "Balanced" if temp == 1 else "Creative"}')
    prob_str = ', '.join([f'{p:.3f}' for p in probs])
    print(f"Temperature = {temp:4.1f} → Probs: [{prob_str}] → {behavior}")

=== EFFECT OF TEMPERATURE ON SAMPLING ===

Raw logits: [2.0, 1.5, 0.5, 0.3, 0.1]
Vocabulary: ['wheat', 'seed', 'crop', 'field', 'soil']

Temperature =  0.1 → Probs: [0.993, 0.007, 0.000, 0.000, 0.000] → Very deterministic
Temperature =  0.5 → Probs: [0.650, 0.240, 0.033, 0.022, 0.015] → → Focused
Temperature =  1.0 → Probs: [0.404, 0.245, 0.090, 0.074, 0.060] → → Balanced
Temperature =  1.5 → Probs: [0.326, 0.237, 0.117, 0.103, 0.091] → → Creative
Temperature =  2.0 → Probs: [0.285, 0.228, 0.135, 0.122, 0.111] → Very random


In [7]:
# Visualization: Temperature Effect
fig, axes = plt.subplots(1, len(temperatures), figsize=(14, 5), sharey=True)

colors_bar = plt.cm.viridis(np.linspace(0.2, 0.9, len(vocab)))

for idx, temp in enumerate(temperatures):
    probs = softmax_with_temperature(logits, temp)
    bars = axes[idx].bar(vocab, probs, color=colors_bar, edgecolor='white', linewidth=0.8)
    axes[idx].set_title(f'T = {temp}', fontsize=13, fontweight='bold')
    axes[idx].set_ylim(0, 1.05)
    axes[idx].tick_params(axis='x', rotation=45)
    
    # Add value labels
    for bar, p in zip(bars, probs):
        if p > 0.01:
            axes[idx].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                          f'{p:.2f}', ha='center', va='bottom', fontsize=8)

axes[0].set_ylabel('Probability', fontsize=12)
fig.suptitle('Effect of Temperature on Token Probability Distribution', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 6. Text Generation Simulation

Here we simulate how a language model generates text token-by-token using a simple n-gram approach.

In [8]:
print("=== SIMPLE TEXT GENERATION (Bigram Model) ===")

# Simple training corpus related to seeds dataset
training_texts = [
    "wheat seeds are classified into three varieties",
    "wheat kernels have different measurements",
    "seeds are measured by area and perimeter",
    "classification of wheat varieties uses machine learning",
    "machine learning algorithms classify wheat seeds accurately"
]

# Build bigram model
bigram_model = {}
for text in training_texts:
    words = text.lower().split()
    for i in range(len(words) - 1):
        current = words[i]
        next_word = words[i + 1]
        if current not in bigram_model:
            bigram_model[current] = []
        bigram_model[current].append(next_word)

def generate_text(start_word, model, max_length=15):
    """Generate text using bigram model."""
    result = [start_word]
    current = start_word
    for _ in range(max_length):
        if current in model:
            next_word = np.random.choice(model[current])
            result.append(next_word)
            current = next_word
        else:
            break
    return ' '.join(result)

print(f"\nTraining corpus:")
for t in training_texts:
    print(f"  {t}")

print(f"\nGenerated texts (starting with 'wheat'):")
for i in range(5):
    generated = generate_text('wheat', bigram_model)
    print(f"  [{i+1}] {generated}")

=== SIMPLE TEXT GENERATION (Bigram Model) ===

Training corpus:
  wheat seeds are classified into three varieties
  wheat kernels have different measurements
  seeds are measured by area and perimeter
  classification of wheat varieties uses machine learning
  machine learning algorithms classify wheat seeds accurately

Generated texts (starting with 'wheat'):
  [1] wheat kernels have different measurements
  [2] wheat varieties uses machine learning algorithms classify wheat varieties uses machine learning
  [3] wheat varieties uses machine learning algorithms classify wheat seeds are measured by area and perimeter
  [4] wheat seeds accurately
  [5] wheat seeds are classified into three varieties


## 7. Top-k and Top-p (Nucleus) Sampling

In [9]:
print("=== SAMPLING STRATEGIES ===")

# Simulated probability distribution
tokens_vocab = ['wheat', 'seed', 'crop', 'field', 'soil', 'rain', 'sun']
probs_dist = np.array([0.35, 0.25, 0.15, 0.12, 0.08, 0.03, 0.02])

print(f"\nOriginal probabilities:")
for t, p in zip(tokens_vocab, probs_dist):
    print(f"  {t:<5}: {p:.3f}")

# Top-k Sampling
k = 3
print(f"\n--- Top-k Sampling (k={k}) ---")
top_k_mask = np.zeros_like(probs_dist)
top_k_indices = np.argsort(probs_dist)[::-1][:k]
top_k_mask[top_k_indices] = probs_dist[top_k_indices]
top_k_probs = top_k_mask / top_k_mask.sum()

print(f"  Only consider top {k} tokens:")
for t, p in zip(tokens_vocab, top_k_probs):
    marker = '✓' if p > 0 else '✗'
    print(f"  {t:<5}: {p:.3f} {marker}")

# Top-p (Nucleus) Sampling
p_threshold = 0.80
print(f"\n--- Top-p (Nucleus) Sampling (p={p_threshold:.2f}) ---")
sorted_indices = np.argsort(probs_dist)[::-1]
cumulative = np.cumsum(probs_dist[sorted_indices])

# Find cutoff
nucleus = set()
for i, idx in enumerate(sorted_indices):
    nucleus.add(idx)
    if cumulative[i] >= p_threshold:
        break

top_p_mask = np.zeros_like(probs_dist)
for idx in nucleus:
    top_p_mask[idx] = probs_dist[idx]
top_p_probs = top_p_mask / top_p_mask.sum() if top_p_mask.sum() > 0 else top_p_mask

print(f"  Include tokens until cumulative prob ≥ {p_threshold:.2f}:")
cum = 0
for i, (t, p) in enumerate(zip(tokens_vocab, probs_dist)):
    cum += p
    marker = '✓' if i in nucleus else '✗'
    print(f"  {t:<5}: {top_p_probs[i]:.3f} {marker} (cumul: {cum:.3f})")

=== SAMPLING STRATEGIES ===

Original probabilities:
  wheat: 0.350
  seed : 0.250
  crop : 0.150
  field: 0.120
  soil : 0.080
  rain : 0.030
  sun  : 0.020

--- Top-k Sampling (k=3) ---
  Only consider top 3 tokens:
  wheat: 0.467 ✓
  seed : 0.333 ✓
  crop : 0.200 ✓
  field: 0.000 ✗
  soil : 0.000 ✗
  rain : 0.000 ✗
  sun  : 0.000 ✗

--- Top-p (Nucleus) Sampling (p=0.80) ---
  Include tokens until cumulative prob ≥ 0.80:
  wheat: 0.438 ✓ (cumul: 0.350)
  seed : 0.312 ✓ (cumul: 0.600)
  crop : 0.188 ✓ (cumul: 0.750)
  field: 0.062 ✓ (cumul: 0.870)
  soil : 0.000 ✗ (cumul: 0.950)
  rain : 0.000 ✗ (cumul: 0.980)
  sun  : 0.000 ✗ (cumul: 1.000)


## 8. Applications of Generative AI

In [10]:
# Visualization: Applications of Generative AI
applications = {
    'Text Generation': 85,
    'Code Generation': 78,
    'Image Generation': 82,
    'Chatbots & Assistants': 90,
    'Summarization': 75,
    'Translation': 70,
    'Data Augmentation': 65,
    'Drug Discovery': 55,
    'Music Composition': 50,
    'Video Generation': 45
}

fig, ax = plt.subplots(figsize=(12, 7))

apps = list(applications.keys())
scores = list(applications.values())

# Sort by score
sorted_pairs = sorted(zip(scores, apps), reverse=True)
scores_sorted = [s for s, _ in sorted_pairs]
apps_sorted = [a for _, a in sorted_pairs]

colors = plt.cm.coolwarm(np.linspace(0.15, 0.85, len(apps_sorted)))
bars = ax.barh(apps_sorted, scores_sorted, color=colors, edgecolor='white', linewidth=0.8, height=0.7)

for bar, score in zip(bars, scores_sorted):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
            f'{score}%', va='center', fontsize=11, fontweight='bold')

ax.set_xlabel('Adoption / Impact Score (%)', fontsize=13)
ax.set_title('Generative AI — Key Application Areas & Impact', fontsize=15, fontweight='bold')
ax.set_xlim(0, 100)
ax.invert_yaxis()

plt.tight_layout()
plt.show()

## 9. Prompt Design Best Practices

In [11]:
print("=== PROMPT DESIGN: GOOD vs BAD EXAMPLES ===")

examples = [
    {
        "bad": "Tell me about wheat.",
        "good": """Describe the three main varieties of wheat (Kama, Rosa, Canadian)
     found in the Seeds dataset. For each variety, list 2-3 distinguishing
     physical characteristics based on kernel measurements.""",
        "reason": "Specific scope, clear structure, measurable output"
    },
    {
        "bad": "Analyze data.",
        "good": """Given a dataset with 7 features (area, perimeter, compactness,
     kernel length, kernel width, asymmetry coefficient, groove length),
     suggest the top 3 most important features for classifying wheat
     varieties. Explain your reasoning.""",
        "reason": "Provides context, specific number, asks for reasoning"
    },
    {
        "bad": "Write code.",
        "good": """Write a Python function that takes a pandas DataFrame of wheat seed
     measurements and returns the accuracy of a Random Forest classifier
     using 5-fold cross-validation. Use scikit-learn.""",
        "reason": "Specifies language, library, method, and expected output"
    }
]

for ex in examples:
    print(f"\n{'━' * 64}")
    print(f"\n  ❌ BAD PROMPT:")
    print(f"     {ex['bad']}")
    print(f"\n  ✅ GOOD PROMPT:")
    print(f"     {ex['good']}")
    print(f"\n  💡 WHY BETTER: {ex['reason']}")

=== PROMPT DESIGN: GOOD vs BAD EXAMPLES ===

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  ❌ BAD PROMPT:
     Tell me about wheat.

  ✅ GOOD PROMPT:
     Describe the three main varieties of wheat (Kama, Rosa, Canadian)
     found in the Seeds dataset. For each variety, list 2-3 distinguishing
     physical characteristics based on kernel measurements.

  💡 WHY BETTER: Specific scope, clear structure, measurable output

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  ❌ BAD PROMPT:
     Analyze data.

  ✅ GOOD PROMPT:
     Given a dataset with 7 features (area, perimeter, compactness,
     kernel length, kernel width, asymmetry coefficient, groove length),
     suggest the top 3 most important features for classifying wheat
     varieties. Explain your reasoning.

  💡 WHY BETTER: Provides context, specific number, asks for reasoning

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  ❌ BAD PROMPT:
     Write code.

  ✅ GOOD PROM

## 10. Key Generative AI Parameters Summary

In [12]:
print("=== KEY PARAMETERS IN GENERATIVE AI MODELS ===")

params_table = [
    ('Temperature', '0.0–2.0', 'Controls randomness. Low=deterministic, High=creative'),
    ('Top-k', '1–100', 'Limits sampling to top k tokens'),
    ('Top-p (nucleus)', '0.0–1.0', 'Samples from smallest set with cumulative prob ≥ p'),
    ('Max tokens', '1–4096+', 'Maximum number of tokens to generate'),
    ('Frequency penalty', '0.0–2.0', 'Penalizes repeated tokens'),
    ('Presence penalty', '0.0–2.0', 'Encourages topic diversity'),
    ('Stop sequences', 'Strings', 'Tokens that signal generation should stop'),
]

print(f"\n┌{'─'*18}┬{'─'*10}┬{'─'*50}┐")
print(f"│ {'Parameter':<16} │ {'Range':<8} │ {'Description':<48} │")
print(f"├{'─'*18}┼{'─'*10}┼{'─'*50}┤")
for name, range_val, desc in params_table:
    print(f"│ {name:<16} │ {range_val:<8} │ {desc:<48} │")
print(f"└{'─'*18}┴{'─'*10}┴{'─'*50}┘")

=== KEY PARAMETERS IN GENERATIVE AI MODELS ===

┌──────────────────┬──────────┬──────────────────────────────────────────────────┐
│ Parameter        │ Range    │ Description                                      │
├──────────────────┼──────────┼──────────────────────────────────────────────────┤
│ Temperature      │ 0.0–2.0  │ Controls randomness. Low=deterministic, High=creative │
│ Top-k            │ 1–100    │ Limits sampling to top k tokens                  │
│ Top-p (nucleus)  │ 0.0–1.0  │ Samples from smallest set with cumulative prob ≥ p │
│ Max tokens       │ 1–4096+  │ Maximum number of tokens to generate             │
│ Frequency penalty│ 0.0–2.0  │ Penalizes repeated tokens                        │
│ Presence penalty │ 0.0–2.0  │ Encourages topic diversity                       │
│ Stop sequences   │ Strings  │ Tokens that signal generation should stop         │
└──────────────────┴──────────┴──────────────────────────────────────────────────┘


## 11. Conclusion

In this lab, we explored the fundamentals of Generative AI and prompt engineering:

1. **Generative AI Overview** — Models that create new content by learning patterns from data
2. **Key Model Types** — GANs, VAEs, Transformers, and Diffusion Models
3. **Transformer Architecture** — Self-attention mechanism enables understanding of contextual relationships
4. **Prompt Engineering Techniques:**
   - **Zero-shot** — Direct instructions without examples
   - **Few-shot** — Learning from a few examples in the prompt
   - **Chain-of-Thought** — Step-by-step reasoning
   - **Role-based** — Assigning expert personas
   - **Instruction-based** — Structured, specific instructions
5. **Temperature & Sampling** — Parameters that control output diversity and creativity
6. **Top-k & Top-p Sampling** — Strategies to balance quality and diversity
7. **Best Practices** — Specificity, context, and structured prompts yield better results

**Key Takeaways:**
- Generative AI is powered by large-scale neural networks (especially Transformers)
- Prompt engineering is crucial for extracting high-quality outputs from language models
- Temperature controls the trade-off between deterministic and creative outputs
- Well-designed prompts should be specific, structured, and contextually rich
- Understanding sampling strategies (Top-k, Top-p) helps in fine-tuning generation quality